# 7B Block-Design Analysis

**Data:** each run consists of 20 emotional prompts followed by 20 neutral prompts (or vice versa), separated by a full node reboot.

**Pipeline:**
1. Clustering on **raw** features — single-feature sweep, majority filter, MWU with Bonferroni.
2. Check if `elapsed_ms` differs significantly between conditions. Residualise only if it does.
3. Clustering on **residualised** features (if applicable).
4. **Null test:** random label assignment within a single condition, checked across 20 repeats. Accuracy should hover around 50%.

**Note on residualisation:** `prompt_index` is NOT used as a covariate — in the block design, emotional and neutral trials occupy different block positions, so prompt_index is fully correlated with condition label and cannot serve as a confound control.

## §1 — Imports and configuration

In [59]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR   = Path.home() / 'Desktop' / 'mccviahat'
DATA_DIR   = BASE_DIR / 'data' / 'shorter7b'
RESULT_DIR = BASE_DIR / 'results' / 'hat_analysis'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

assert BASE_DIR.exists(), f'Repo root not found: {BASE_DIR}'

# ── Runs to load ───────────────────────────────────────────────────────────────
RUNS = ['187a', '187b', '194a', '194b', '200a', '200b', '204a', '204b']

# ── L1 indicators ──────────────────────────────────────────────────────────────
L1_INDICATORS = ['core_power.throttle', 'hat_TLB', 'tlb:tlb_flush']

# ── Analysis config ────────────────────────────────────────────────────────────
MAJORITY_THRESHOLD = 0.75
N_INIT             = 50
RAND_SEED          = 42
N_NULL_REPEATS     = 20

print('Configuration OK')
print(f'  Runs              : {RUNS}')
print(f'  Majority threshold: {MAJORITY_THRESHOLD:.0%}')
print(f'  Null repeats      : {N_NULL_REPEATS}')

Configuration OK
  Runs              : ['187a', '187b', '194a', '194b', '200a', '200b', '204a', '204b']
  Majority threshold: 75%
  Null repeats      : 20


## §2 — Load data

In [60]:
dfs = []
runs_loaded = []
for run in RUNS:
    for stem in [f'blocks{run}', run]:
        p = DATA_DIR / f'{stem}.csv'
        if p.exists():
            d = pd.read_csv(p)
            d['run'] = run
            dfs.append(d)
            runs_loaded.append(run)
            break
    else:
        print(f'  WARNING: no CSV found for run {run}')

assert dfs, 'No data loaded. Check DATA_DIR and RUNS.'
df_all = pd.concat(dfs, ignore_index=True)
df_all['label'] = (df_all['condition'] == 'emotional').astype(int)

print(f'Loaded {len(runs_loaded)} runs: {runs_loaded}')
print(f'  Total trials : {len(df_all)}')
print(f'  Emotional    : {(df_all.condition == "emotional").sum()}')
print(f'  Neutral      : {(df_all.condition == "neutral").sum()}')

Loaded 8 runs: ['187a', '187b', '194a', '194b', '200a', '200b', '204a', '204b']
  Total trials : 320
  Emotional    : 160
  Neutral      : 160


## §3 — Select L1 features

In [61]:
def indicator_of(col):
    for ind in L1_INDICATORS:
        if col.startswith(ind + '__'):
            return ind
    return ''

l1_cols = [c for c in df_all.columns if indicator_of(c) in L1_INDICATORS]

X_l1 = df_all[["core_power.throttle__slope", "core_power.throttle__variance", "core_power.throttle__spectral_entropy", "core_power.throttle__mean_rate", "core_power.throttle__lz_complexity"]].copy()
X_l1 = X_l1.dropna(axis=1, how='all')
X_l1 = X_l1.fillna(X_l1.median())
X_l1 = X_l1.loc[:, X_l1.std() > 0]

print(f'L1 features available: {X_l1.shape[1]}')
print(f'Features: {list(X_l1.columns)}')

L1 features available: 5
Features: ['core_power.throttle__slope', 'core_power.throttle__variance', 'core_power.throttle__spectral_entropy', 'core_power.throttle__mean_rate', 'core_power.throttle__lz_complexity']


## §4 — Helper functions

In [62]:
def kmeans_acc_ari(X, y, n_init=N_INIT, seed=RAND_SEED):
    """K-means (k=2) with majority-vote alignment. Returns (accuracy, ARI)."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    km = KMeans(n_clusters=2, n_init=n_init, random_state=seed)
    pred = km.fit_predict(Xs)
    cm = confusion_matrix(y, pred)
    if cm.shape != (2, 2):
        return 0.5, 0.0
    acc = max(
        (cm[0,0] + cm[1,1]) / cm.sum(),
        (cm[0,1] + cm[1,0]) / cm.sum()
    )
    ari = adjusted_rand_score(y, pred)
    return float(acc), float(ari)


def single_feature_sweep(X_feat_df, df, label_col='label'):
    """Single-feature sweep with per-run direction analysis and majority filter."""
    y_true = df[label_col].values
    runs_unique = df['run'].unique()
    records = []
    for col in X_feat_df.columns:
        acc, ari = kmeans_acc_ari(X_feat_df[[col]].values, y_true)
        e_mean = X_feat_df.loc[df['condition'] == 'emotional', col].mean()
        n_mean = X_feat_df.loc[df['condition'] == 'neutral',   col].mean()
        direction = '↑E' if e_mean > n_mean else '↑N'
        
        run_dirs = []
        for run in runs_unique:
            mask_run = df['run'] == run
            e_run = X_feat_df.loc[mask_run & (df['condition'] == 'emotional'), col].mean()
            n_run = X_feat_df.loc[mask_run & (df['condition'] == 'neutral'),   col].mean()
            if not (np.isnan(e_run) or np.isnan(n_run)):
                run_dirs.append('↑E' if e_run > n_run else '↑N')
        
        n_majority = sum(1 for d in run_dirs if d == direction)
        n_valid    = len(run_dirs)
        maj_frac   = n_majority / n_valid if n_valid > 0 else 0.0
        
        records.append({
            'feature':       col,
            'accuracy':      acc,
            'ari':           ari,
            'direction':     direction,
            'runs_majority': f'{n_majority}/{n_valid}',
            'majority_frac': maj_frac,
        })
    return pd.DataFrame(records).sort_values('accuracy', ascending=False).reset_index(drop=True)


def mwu_bonferroni(X_feat_df, df, features):
    """Mann-Whitney U with Bonferroni correction for a list of features."""
    if not len(features):
        return pd.DataFrame()
    records = []
    for feat in features:
        e = X_feat_df.loc[df.condition == 'emotional', feat].dropna()
        n = X_feat_df.loc[df.condition == 'neutral',   feat].dropna()
        u, p_raw = mannwhitneyu(e, n, alternative='two-sided')
        p_corr = min(p_raw * len(features), 1.0)
        records.append({
            'feature':     feat,
            'U':           u,
            'p_raw':       p_raw,
            'p_corrected': p_corr,
            'significant': 'YES' if p_corr < 0.05 else 'NO',
        })
    return pd.DataFrame(records)

print('Helpers defined.')

Helpers defined.


## §5 — Raw clustering (no residualisation)

In [63]:
print('=' * 70)
print('RAW CLUSTERING (no residualisation)')
print('=' * 70)

sf_raw = single_feature_sweep(X_l1, df_all)

print('\nAll features (sorted by accuracy):')
print(sf_raw[['feature', 'accuracy', 'ari', 'direction', 'runs_majority', 'majority_frac']].to_string(index=False))

sf_raw_filtered = sf_raw[sf_raw['majority_frac'] >= MAJORITY_THRESHOLD].reset_index(drop=True)
print(f'\nFeatures passing ≥{MAJORITY_THRESHOLD:.0%} majority: {len(sf_raw_filtered)}')
if len(sf_raw_filtered):
    print(sf_raw_filtered[['feature', 'accuracy', 'ari', 'direction', 'runs_majority']].to_string(index=False))

print('\nMann-Whitney U (Bonferroni-corrected across majority-passing features):')
mwu_raw = mwu_bonferroni(X_l1, df_all, sf_raw_filtered['feature'].tolist())
if not mwu_raw.empty:
    print(mwu_raw.to_string(index=False))
else:
    print('  No features passed majority threshold.')

RAW CLUSTERING (no residualisation)

All features (sorted by accuracy):
                              feature  accuracy       ari direction runs_majority  majority_frac
           core_power.throttle__slope  0.543750  0.004553        ↑N           6/8          0.750
        core_power.throttle__variance  0.531250  0.000823        ↑N           6/8          0.750
   core_power.throttle__lz_complexity  0.528125  0.000039        ↑N           5/8          0.625
core_power.throttle__spectral_entropy  0.500000 -0.003105        ↑N           4/8          0.500
       core_power.throttle__mean_rate  0.500000 -0.003145        ↑N           5/8          0.625

Features passing ≥75% majority: 2
                      feature  accuracy      ari direction runs_majority
   core_power.throttle__slope   0.54375 0.004553        ↑N           6/8
core_power.throttle__variance   0.53125 0.000823        ↑N           6/8

Mann-Whitney U (Bonferroni-corrected across majority-passing features):
                   

## §8 — Null test: random labels within a single condition

Clustering on emotional-only trials (or neutral-only trials) with randomly assigned pseudo-labels. Accuracy should hover around 50% and ARI around 0, confirming there is no spurious structure being picked up.

In [64]:
# Pick the best raw feature to test for spurious structure
NULL_FEATURE = sf_raw.iloc[0]['feature']
print(f'Null test feature: {NULL_FEATURE}')
print(f'(Testing whether clustering picks up structure with random labels)\n')

rng = np.random.default_rng(RAND_SEED)

for cond in ['emotional', 'neutral']:
    sub = df_all[df_all.condition == cond].copy().reset_index(drop=True)
    n_total = len(sub)
    X_sub = sub[[NULL_FEATURE]].fillna(sub[[NULL_FEATURE]].median()).values
    
    accs, aris = [], []
    for _ in range(N_NULL_REPEATS):
        # Assign exactly 50/50 random pseudo-labels
        pseudo = np.zeros(n_total, dtype=int)
        pseudo[:n_total // 2] = 1
        rng.shuffle(pseudo)
        
        if len(np.unique(pseudo)) < 2:
            continue
        
        acc, ari = kmeans_acc_ari(X_sub, pseudo)
        accs.append(acc)
        aris.append(ari)
    
    print(f'{cond:10s}  n={n_total:4d}  '
          f'mean_acc={np.mean(accs):.4f} ± {np.std(accs):.4f}  '
          f'mean_ARI={np.mean(aris):.4f} ± {np.std(aris):.4f}')

print(f'\nExpected: accuracy ≈ 0.50, ARI ≈ 0.00')
print(f'Repeats per condition: {N_NULL_REPEATS}')

Null test feature: core_power.throttle__slope
(Testing whether clustering picks up structure with random labels)

emotional   n= 160  mean_acc=0.5356 ± 0.0289  mean_ARI=0.0023 ± 0.0131
neutral     n= 160  mean_acc=0.5275 ± 0.0233  mean_ARI=-0.0011 ± 0.0078

Expected: accuracy ≈ 0.50, ARI ≈ 0.00
Repeats per condition: 20


## §9 — Summary

In [65]:
print('=' * 70)
print('SUMMARY')
print('=' * 70)
print(f'Runs loaded          : {len(runs_loaded)}')
print(f'Total trials         : {len(df_all)}')
print(f'Features tested      : {X_l1.shape[1]}')
print()
print(f'Raw clustering — best feature      : {sf_raw.iloc[0].feature}')
print(f'                  accuracy         : {sf_raw.iloc[0].accuracy:.4f}')
print(f'                  ARI              : {sf_raw.iloc[0].ari:.4f}')
print(f'                  direction        : {sf_raw.iloc[0].direction} ({sf_raw.iloc[0].runs_majority})')
print()
print(f'elapsed_ms condition difference: p={p_ms:.4f}')


SUMMARY
Runs loaded          : 8
Total trials         : 320
Features tested      : 5

Raw clustering — best feature      : core_power.throttle__slope
                  accuracy         : 0.5437
                  ARI              : 0.0046
                  direction        : ↑N (6/8)

elapsed_ms condition difference: p=0.3735
